## Task 2: LoRA Fine-Tuning

In this task, I fine-tuned GPT2 using LoRA (PEFT) on a subset of the IMDB dataset.

Key Design Choices:
- Used LoRA for efficient fine-tuning instead of updating full model weights
- Limited dataset size for faster experimentation
- Used short sequence length (128 tokens) to reduce memory usage

Observations:
- The fine-tuned model adapts slightly to sentiment-related text
- LoRA significantly reduces the number of trainable parameters
- Training is efficient even on limited hardware

This demonstrates practical and scalable LLM fine-tuning.

In [1]:
!pip install transformers datasets peft accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

In [3]:
dataset = load_dataset("imdb", split="train[:1000]")

print("Sample data:\n")
print(dataset[0])

Sample data:

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are 

In [4]:
model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully!


In [5]:
tokenizer.pad_token = tokenizer.eos_token

In [6]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("LoRA applied successfully!")
model.print_trainable_parameters()

LoRA applied successfully!
trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


C:\Users\Divyasri\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [7]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    
    tokens["labels"] = tokens["input_ids"].copy()  # 🔥 IMPORTANT LINE
    
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=20,
    save_strategy="no"
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [10]:
trainer.train()

C:\Users\Divyasri\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,4.230145
40,4.254597
60,4.069381
80,4.086358
100,4.012692
120,4.303673
140,3.979226
160,4.004152
180,4.120120
200,4.193742


TrainOutput(global_step=500, training_loss=4.047309692382813, metrics={'train_runtime': 1282.3926, 'train_samples_per_second': 0.78, 'train_steps_per_second': 0.39, 'total_flos': 65549500416000.0, 'train_loss': 4.047309692382813, 'epoch': 1.0})

In [11]:
model.save_pretrained("lora-gpt2")
tokenizer.save_pretrained("lora-gpt2")

print("Model saved!")

Model saved!


In [12]:
# ============================================
# BEFORE vs AFTER FINE-TUNING COMPARISON
# ============================================

# Load base (original) GPT2 model
base_model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

# Set both models to evaluation mode
model.eval()
base_model.eval()

# Input prompt
prompt = "This movie is"

inputs = tokenizer(prompt, return_tensors="pt")

# BEFORE fine-tuning
output_before = base_model.generate(
    **inputs,
    max_length=50
)

# AFTER fine-tuning
output_after = model.generate(
    **inputs,
    max_length=50
)

# Print outputs
print("===== BEFORE FINE-TUNING =====\n")
print(tokenizer.decode(output_before[0], skip_special_tokens=True))

print("\n===== AFTER FINE-TUNING =====\n")
print(tokenizer.decode(output_after[0], skip_special_tokens=True))

# Insight
print("\n🔍 Insight:")
print("The fine-tuned model shows slight adaptation towards sentiment-based text compared to the base model.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


===== BEFORE FINE-TUNING =====

This movie is a great example of how to make a movie that is not just a movie, but a movie that is a movie. It's a movie that is a movie that is a movie that is a movie that is a movie that is a

===== AFTER FINE-TUNING =====

This movie is a great example of how to make a movie that is not only entertaining but also entertaining. It is a great example of how to make a movie that is not only entertaining but also entertaining. It is a great example of how to make

🔍 Insight:
The fine-tuned model shows slight adaptation towards sentiment-based text compared to the base model.


### Observations

- The fine-tuned model shows slight adaptation toward sentiment-related text generation.
- Compared to the base GPT2 model, the output demonstrates more structured and context-aware phrasing.
- Training was efficient due to LoRA, with only a small fraction of parameters updated.
- Even with a small dataset and limited epochs, noticeable behavioral changes were observed.

---

### Key Insight

LoRA enables efficient fine-tuning of large language models by updating only a subset of parameters, making it highly suitable for resource-constrained environments while still achieving meaningful performance improvements.